In [1]:
import numpy as np
from iminuit import Minuit
from iminuit.cost import ExtendedBinnedNLL
from scipy.stats import expon, norm, uniform, chi2
from collections.abc import Callable
import plotly.graph_objects as go
import inspect


def exp( x , N ,a, A, tau):
    return a*N*expon.cdf(x , A , tau) 

def sturges( N: int):
    return int(np.round( 1 + 3.322 * np.log10(N)))

def exp_unif(x, a, b, tau, A):
    return a * N * (expon.cdf(x, A, tau) + b * N * uniform.cdf(x, 0, 8e-6))

def gauss_cdf(x, N, mu, sigma):
    return N * norm.cdf(x, mu, sigma)
def exp_fit(cdf, a, A, tau, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, A, tau )
    n.fixed["A"] = True
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def exp_unif_fit(cdf, a, b, tau, A, count , edges):
    
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, a, b, tau, A )
    # n.fixed['N'] = True
    n.fixed['A'] = True
    # n.limits['b'] = (0, 1)
    n.migrad(ncall = 1000000)
    n.hesse()
    return n

def gauss_fit( cdf , N , mu , sigma , count , edges):
    cost = ExtendedBinnedNLL(count, edges, cdf)
    n = Minuit(cost, mu , sigma )
    n.migrad(ncall = 1000000)
    n.hesse()
    return n


#! Funzione per generalizzare la funzione usando la lunghezza del dataset
def function_generator_with_variable_N(func: Callable, N: int):
    sig = inspect.signature(func)
    
    if "x" not in sig.parameters or "N" not in sig.parameters:
        raise SyntaxError("function defined wrong: it needs an x as first parameter and N as other parameter")
    def wrapper(x, *args, **kwargs):
        return func(x, N, *args, **kwargs)

    wrapper.__signature__ = inspect.Signature(
        [inspect.Parameter('x', inspect.Parameter.POSITIONAL_OR_KEYWORD)] +
        list(inspect.signature(func).parameters.values())[2:]
    )


    return wrapper

def dataset_analysis(dataset: np.ndarray , creator: Callable, bins: int , args: dict) -> Minuit:
    model_function = function_generator_with_variable_N( creator , len(dataset))
    count, edges = np.histogram( dataset , bins=bins) 
    cost = ExtendedBinnedNLL(count, edges, model_function)

    minuit_element =  Minuit(cost, *args.values())
    
    if "A" in args.keys():
        minuit_element.fixed["A"] = True

    return minuit_element

def end(m:Minuit) -> None:
    m.migrad()
    m.hesse()
    display(m)

In [22]:
data = np.genfromtxt("Data/timestamp/error_prop.csv")
data_700 = np.genfromtxt("Data/timestamp/error_prop_700ns.csv")
data_26_02 = np.genfromtxt("Data/timestamp/error_prop_26_02.csv")
print(len(data))

240203


In [25]:
nbin = 10
count, edges = np.histogram( data , bins = 7, density=False)
count1, edges1 = np.histogram(data_700, bins = 5)
count2, edges2 = np.histogram(data_26_02, bins = 10)
fig = go.Figure()
# fig.add_trace(go.Bar(x=edges[:-1],  y=count,   name='error1', width=np.diff(edges)))
# fig.add_trace(go.Bar(x= edges1, y = count1, name = '700ns', width = np.diff(edges1)))
fig.add_trace(go.Bar(x= edges2, y = count2, name = '26_02', width = np.diff(edges2)))
fig.update_layout(bargap = 0)

In [21]:
data = data[data > 6.69e-6]
count, edges = np.histogram( data , bins = 7, density=False)
N = len(data)
n = Minuit(ExtendedBinnedNLL(count, edges, gauss_cdf),N, 6.708e-6 , 0.05e-7)
n.fixed["N"] = True
n.hesse()
# n.interactive()

┌─────────────────────────────────────────────────────────────────────────┐
│                               External                                  │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 3.155e+04 (χ²/ndof = 6310.5)│              Nfcn = 23               │
│ EDM = 1.64e+04 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│         INVALID Minimum          │   ABOVE EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ N     │  240.2e3  │   2.4e3   │            │            │         │         │  yes  │
│ 1 │ mu    │6.708000e-6│0.000011e-6│            │            │         │         │       │
│ 2 │ sigma │ 5.000e-9  │ 0.013e-9  │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────────────────┐
│       │         N        mu     sigma │
├───────┼───────────────────────────────┤
│     N │         0         0         0 │
│    mu │         0  1.25e-22 -0.02e-21 │
│ sigma │         0 -0.02e-21  1.75e-22 │
└───────┴───────────────────────────────┘